# Vocabulary Extension Strategy and Documentation

Given a llama 3 base tokenizer pretrained mainly on high resource langauges we want to extned the vocabulary of the tokeniezr so taht its appropriate for our huge multilingual corpus

However we dont want to extend the vocab in every langauge. E.g we dont need to add english vocabs since the tokenizer is already wenn suited for english data.

Therefore we first identify which languges we need to extend the tokenizer for.
### Step 1: Detect New (Unknown) Languages
We test how well the base tokenizer handles well the base tokenizer handles all the languages.

We run the tokenizer on a sample of each language and collect:
- **Characters per token**
- **Pieces per word**
- **Unknown token rate**

Languages showing high fragmentation or many unknown tokens are considered *new or poorly covered* and become candidates for vocabulary extension.
For this step the script compute_tokenizer_coverage.py was written, it extracts the output in .metric/tokenizer_metric.csvimport pandas as pd




In [2]:
import pandas as pd
df = pd.read_csv("/scratch/anonymous-project/moe-study/data_prep/metrics/tokenizer_metrics_llama3.2-1b_all_.csv")
print("Best covered languages:")
print(df.nsmallest(15, "pieces_per_word")[["language", "pieces_per_word", "chars_per_token", "unknown_rate"]])

print("\n Worst covered languages:")
print(df.nlargest(5, "pieces_per_word")[["language", "pieces_per_word", "chars_per_token", "unknown_rate"]])

Best covered languages:
     language  pieces_per_word  chars_per_token  unknown_rate
197   english         1.312877         4.517402           0.0
195  spa_Latn         1.619902         3.740534           0.0
191  fas_Arab         1.658605         3.029692           0.0
100  nin_Latn         1.709528         2.516823           0.0
54   ile_Latn         1.717754         3.425544           0.0
78   lbb_Latn         1.782787         2.691116           0.0
32   csy_Latn         1.807866         2.718838           0.0
161  hin_Latn         1.815506         2.886623           0.0
139  xmm_Latn         1.866019         2.983659           0.0
106  jbo_Latn         1.878802         2.577222           0.0
115  nii_Latn         1.918645         2.618079           0.0
26   ibg_Latn         1.997961         3.040667           0.0
126  whg_Latn         2.000003         2.813174           0.0
141  mww_Latn         2.005804         2.474800           0.0
77   lgl_Latn         2.008272         2.47729

The unknown token analysis failed, since llama3 tokenizer doesnt have a unknown token:

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B", use_fast=True)
print("unk_token:", tokenizer.unk_token)
print("unk_token_id:", tokenizer.unk_token_id)



/scratch/anonymous-project/htyllm-pg/anonymous-user/conda/envs/moe-study/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


unk_token: None
unk_token_id: None


In [4]:
tokenizer.unk_token = "<unk>"
print("unk_token:", tokenizer.unk_token)

text = "This is a test sentence with some unknownwordthatdoesnotexist."
text2 = "This text has invalid UTF-8 bytes: \xC3\x28\xA0\xA1"
text3 = "A‏͏B‍C͟D̷̹̰̺͘E‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍‍"
tokenizer.tokenize(text)
tokenizer.tokenize(text2)
tokenizer.tokenize(text3)
print("Tokenizing some text with unknown words and invalid UTF-8 bytes:")
print(tokenizer.tokenize(text))
print(tokenizer.tokenize(text2))
print(tokenizer.tokenize(text3))

unk_token: <unk>
Tokenizing some text with unknown words and invalid UTF-8 bytes:
['This', 'Ġis', 'Ġa', 'Ġtest', 'Ġsentence', 'Ġwith', 'Ġsome', 'Ġunknown', 'word', 'that', 'does', 'not', 'exist', '.']
['This', 'Ġtext', 'Ġhas', 'Ġinvalid', 'ĠUTF', '-', '8', 'Ġbytes', ':', 'ĠÃĥ', '(', 'Âł', 'Â¡']
['A', 'âĢı', 'Í', 'ı', 'B', 'âĢį', 'C', 'Í', 'Ł', 'D', 'Ì', '·', 'Ì', '¹', 'Ì', '°', 'Í', 'ĺ', 'Ì', 'º', 'E', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį', 'âĢį',

I also tried to include multilingual evaluation benchmarks such as Belebele and HellaSwag. However, I decided not to, since many languages were not covered, the evaluation process was very time-consuming, and the language mapping (e.g., eng_Latn → English) between datasets required significant manual work. Additionally, the LM evaluation framework didn’t support some of the languages I wanted — for example, Belebele only provided 31 languages instead of 122.

### Extend of vocabulary extension for each language?
We aim to expand the LLaMA-3 tokenizer from its original 128K vocabulary to 250K tokens, resulting in an additional 122K subword units.
Rather than distributing these new tokens uniformly, we allocate them proportionally to each language’s tokenizer performance.
Languages that exhibit higher fragmentation or lower coverage (i.e., where the base tokenizer performs poorly) receive a larger share of new tokens, while well-covered languages receive fewer or none.
This adaptive allocation ensures that the extended vocabulary grows efficiently and improves coverage where it is most needed.

### Vocabulary extension ranking-based
We expand LLaMA-3 from **128K → 250K** (+**122K** tokens).
Each language gets tokens **proportional to a continuous “inefficiency” score** derived from:
- higher = worse: *pieces per word* (PPW)
- lower = worse: *characters per token* (CPT)
- (optional) *unknown rate* (UNK; here mostly 0 so weight = 0)

We emphasize under-covered languages by raising the score to a power γ>1.

In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

TOTAL_NEW_TOKENS = 122_000
W_PPW, W_CPT, W_UNK = 0.5, 0.5, 0.0
GAMMA = 2.0                            # nonlinearity bigger than one emphasizes on bad languages
CSV_PATH = "/scratch/anonymous-project/moe-study/data_prep/metrics/tokenizer_metrics_llama3.2-1b_all_.csv"
df = pd.read_csv(CSV_PATH)
df.head()


,language,chars_per_token,pieces_per_word,unknown_rate,num_texts
0,bwq_Latn,1.881650,2.816314,0.0,14603
1,dty_Deva,1.745320,3.561404,0.0,18192
2,ayo_Latn,2.480993,2.273708,0.0,14603
3,cho_Latn,2.191390,2.988211,0.0,21234
4,enx_Latn,2.199687,3.271708,0.0,16775


In [6]:
# normalize metrics
scaler = MinMaxScaler()
dfn = df.copy()
dfn["ppw_norm"] = scaler.fit_transform(dfn[["pieces_per_word"]])
dfn["cpt_norm"] = 1 - scaler.fit_transform(dfn[["chars_per_token"]])
dfn["unk_norm"] = scaler.fit_transform(dfn[["unknown_rate"]])
dfn[["language", "ppw_norm", "cpt_norm", "unk_norm"]].head()


,language,ppw_norm,cpt_norm,unk_norm
0,bwq_Latn,0.040174,0.642849,0.0
1,dty_Deva,0.060083,0.676100,0.0
2,ayo_Latn,0.025675,0.496672,0.0
3,cho_Latn,0.044767,0.567305,0.0
4,enx_Latn,0.052342,0.565282,0.0


In [7]:
# linear inefficiency score that indicated how inefficient the tokenizer is in a specific language
dfn["I_linear"] = (
    W_PPW * dfn["ppw_norm"] +
    W_CPT * dfn["cpt_norm"] +
    W_UNK * dfn["unk_norm"]
)

print("linear inefficiency scores: ")
dfn[["language", "I_linear"]].head()


linear inefficiency scores: 


,language,I_linear
0,bwq_Latn,0.341512
1,dty_Deva,0.368092
2,ayo_Latn,0.261173
3,cho_Latn,0.306036
4,enx_Latn,0.308812


In [8]:
# Apply nonlinear scaling to emphasize under-covered languages
dfn["I_nonlinear"] = np.power(dfn["I_linear"], GAMMA)
dfn["inefficiency_score"] = MinMaxScaler().fit_transform(dfn[["I_nonlinear"]])

print(f"Applied nonlinear scaling (gamma = {GAMMA}).")
dfn = dfn.sort_values("inefficiency_score", ascending=False)
dfn[["language", "pieces_per_word", "chars_per_token", "inefficiency_score"]].head(10)


Applied nonlinear scaling (gamma = 2.0).


,language,pieces_per_word,chars_per_token,inefficiency_score
155,dzo_Tibt,38.736257,0.535776,1.000000
159,ksw_Mymr,33.332568,0.512968,0.864095
150,lao_Laoo,27.993365,0.563407,0.724107
140,wuu_Hani,23.361391,1.194428,0.504202
176,ory_Orya,15.474431,0.417294,0.489036
179,ojb_Cans,13.632081,0.423856,0.453634
189,kat_Geor,14.815406,0.558206,0.452850
162,sat_Olck,13.106978,0.435029,0.442255
166,tzm_Tfng,12.130776,0.457184,0.421262
196,tel_Telu,13.025482,0.599254,0.414217


In [9]:
#  now we allocate total new tokens proportionally to inefficiency scores
weights = dfn["inefficiency_score"].to_numpy()
total_w = weights.sum()

if total_w == 0:
    dfn["token_alloc"] = 0
else:
    ideal = weights / total_w * TOTAL_NEW_TOKENS
    floor = np.floor(ideal).astype(int)
    remainder = ideal - floor
    shortfall = TOTAL_NEW_TOKENS - floor.sum()
    if shortfall > 0:
        idx = np.argsort(-remainder)[:shortfall]
        floor[idx] += 1
    dfn["token_alloc"] = floor

assert dfn["token_alloc"].sum() == TOTAL_NEW_TOKENS, "Check allocation sum!"
print(f"Sum of allocated tokens: {dfn['token_alloc'].sum()} (target = {TOTAL_NEW_TOKENS})")


Sum of allocated tokens: 122000 (target = 122000)


In [10]:
print("\nbest 15 languages with token allocation:")
print(dfn[["language", "inefficiency_score", "token_alloc"]].head(15))

print("\nworst 10 (best-covered) languages:")
print(dfn[["language", "inefficiency_score", "token_alloc"]].tail(10))



best 15 languages with token allocation:
     language  inefficiency_score  token_alloc
155  dzo_Tibt            1.000000         4553
159  ksw_Mymr            0.864095         3934
150  lao_Laoo            0.724107         3297
140  wuu_Hani            0.504202         2296
176  ory_Orya            0.489036         2227
179  ojb_Cans            0.453634         2066
189  kat_Geor            0.452850         2062
162  sat_Olck            0.442255         2014
166  tzm_Tfng            0.421262         1918
196  tel_Telu            0.414217         1886
110  tig_Ethi            0.412181         1877
53   crl_Cans            0.404356         1841
183  tcy_Knda            0.403578         1838
198  syc_Syrc            0.391562         1783
142  wsg_Telu            0.381945         1739

worst 10 (best-covered) languages:
     language  inefficiency_score  token_alloc
129  pam_Latn            0.042311          193
139  xmm_Latn            0.038918          177
26   ibg_Latn            0.03

In [11]:
dfn.to_csv("metrics/vocab_extension_plan_ranking_based_all_llama_3.2-1b.csv", index=False)
print("saved")

saved


based on this csv we apply the merged vocab extension plan:

    - First we use the multilingual tokenizer to merge subworks and collect tokens considerations with frequencies
    - Here is an exmaple of how that looks like:

In [12]:
df = pd.read_csv("/scratch/anonymous-project/project_data/moe_study/fw_samples/extended_tokenizer/test/fin_Latn/fin_Latn_tokens_freq.csv")
print(df.head())

    token  frequency
0  ĠmyÃ¶s        881
1   Ġovat        755
2  ĠsekÃ¤        592
3   istÃ¤        549
4  Ġmutta        544


now depending in how many tokens we want to include per langauge, we just include the most frequent ones and use the scripts

merge_tokenizer.py and extend_llama_tokenizer.py

we then test the resuting tokenizer again with improved vocabs:

In [13]:
b = pd.read_csv("/scratch/anonymous-project/moe-study/data_prep/metrics/tokenizer_metrics_llama3.2-1b_all_.csv")
a = pd.read_csv("/scratch/anonymous-project/moe-study/data_prep/metrics/test_after_tokenizer_metrics_llama3.2-1b_all_.csv")

m = b.merge(a, on="language", suffixes=("_b","_a"))
m["chars_impr"] = m["chars_per_token_b"] - m["chars_per_token_a"]
m["pieces_impr"] = m["pieces_per_word_b"] - m["pieces_per_word_a"]

print("Best improvements:\n", m.nlargest(10, ["chars_impr","pieces_impr"])[["language","chars_impr","pieces_impr"]])

Best improvements:
      language  chars_impr  pieces_impr
197   english    0.885680    -0.319127
195  spa_Latn    0.466357    -0.235110
104  pui_Latn    0.391513    -0.351030
190  zsm_Latn    0.360805    -0.266699
54   ile_Latn    0.353241    -0.192479
32   csy_Latn    0.259985    -0.188021
139  xmm_Latn    0.253561    -0.171389
161  hin_Latn    0.218720    -0.146116
141  mww_Latn    0.199234    -0.175637
48   hnn_Latn    0.188645    -0.161004


### Comparison to trained multilingual tokenizer

In [14]:
# and now after extending the vocab with 122k new tokens, we re-evaluate the tokenizer:
df_multi = pd.read_csv("/scratch/anonymous-project/moe-study/data_prep/metrics/multilingual_retrained_tokenizer_metrics_llama3.2-1b_all_.csv")
print("Best covered languages:")
print(df_multi.nsmallest(15, "pieces_per_word")[["language", "pieces_per_word", "chars_per_token", "unknown_rate"]])

print("\n Worst covered languages:")
print(df_multi.nlargest(5, "pieces_per_word")[["language", "pieces_per_word", "chars_per_token", "unknown_rate"]])



Best covered languages:
     language  pieces_per_word  chars_per_token  unknown_rate
198  wbm_Latn         1.141592         4.052417           0.0
57   ctp_Latn         1.162080         4.324022           0.0
137  nin_Latn         1.176955         3.656803           0.0
134  wew_Latn         1.177570         4.841270           0.0
98   mie_Latn         1.177976         3.929084           0.0
110  oku_Latn         1.188427         3.705368           0.0
5    ayo_Latn         1.198073         4.708651           0.0
92   lbb_Latn         1.213858         3.953125           0.0
91   mej_Latn         1.218349         4.746673           0.0
101  mww_Latn         1.219048         4.072041           0.0
35   caa_Latn         1.244223         4.443396           0.0
84   mse_Latn         1.252828         4.018232           0.0
144  nii_Latn         1.256239         3.999095           0.0
105  kpw_Latn         1.263276         3.880060           0.0
0    bwq_Latn         1.278920         4.14359

Since the mutlilingual tokenizer was way bettert han the extended one, and the vocab extension was quite time consuming, i for not decided to just use the mutlilingual tokenizer trained on our own data for training

- merge tokenizers 300k self trained with original
- find word spacer, if not whitespaces, spacy and stanca
- try word segmentation using multilingual part of speech tagging model
- 